In [ ]:
!pip install google-cloud-aiplatform

In [1]:
import os
import vertexai
from vertexai.vision_models import MultiModalEmbeddingModel, Video
import torch
from tqdm import tqdm
import time
import pandas as pd
import subprocess
import imageio_ffmpeg
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "temp/gemini_key.json"

PROJECT_ID = "inbound-descent-488618-d3"
LOCATION = "us-central1" 

print("Authenticating with Google Cloud...")
vertexai.init(project=PROJECT_ID, location=LOCATION)

print("Loading Multimodal Foundation Model...")
model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding")
print("Ready to extract!")

Authenticating with Google Cloud...
Loading Multimodal Foundation Model...


/home1/pkandpal/.conda/envs/calm_conflict/lib/python3.11/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Ready to extract!


In [ ]:
MASTER_DATASET = "out/master_dataset.csv"
OUTPUT_TENSORS = "out/gemini_multimodal_embeddings.pt"
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()

In [4]:
def process_single_row(row):
    """Function to process a single video so we can run multiple at once."""
    sample_id = row['sample_id']
    transcript = str(row['transcript'])
    mp4_path = row['mp4_path']
    start_sec = row['start_sec']
    duration = row['duration_sec']
    
    # Create a unique temp file name for this specific thread!
    temp_video_path = f"temp_video/temp_{sample_id}.mp4"
    
    if transcript.strip() == "" or transcript.lower() == "nan":
        transcript = "[SILENCE]"
        
    if not os.path.exists(mp4_path):
        return sample_id, None
        
    try:
        # 1. SLICE
        cmd = [
            FFMPEG_EXE, "-y", "-loglevel", "error", 
            "-ss", str(start_sec), "-i", mp4_path, "-t", str(duration),
            "-c:v", "libx264", "-preset", "ultrafast", "-c:a", "aac",
            temp_video_path
        ]
        subprocess.run(cmd, check=True)
        
        # 2. UPLOAD TO VERTEX AI
        video_input = Video.load_from_file(temp_video_path)
        embeddings = model.get_embeddings(video=video_input, contextual_text=transcript)
        
        # Clean up the temp file
        if os.path.exists(temp_video_path): os.remove(temp_video_path)
            
        if embeddings.video_embeddings:
            fused_vector = embeddings.video_embeddings[0].embedding
            return sample_id, torch.tensor(fused_vector, dtype=torch.float32)
            
    except Exception as e:
        # Clean up temp file on failure
        if os.path.exists(temp_video_path): os.remove(temp_video_path)
        # Sleep briefly in case it's a rate limit error
        time.sleep(2)
        
    return sample_id, None


def extract_vertex_multimodal_fast():
    print(f"Loading {MASTER_DATASET}...")
    df = pd.read_csv(MASTER_DATASET)
    
    # ==========================================
    # CHECKPOINTING: Load existing progress!
    # ==========================================
    if os.path.exists(OUTPUT_TENSORS):
        multimodal_dict = torch.load(OUTPUT_TENSORS)
        print(f"Resuming from checkpoint! Found {len(multimodal_dict)} existing embeddings.")
    else:
        multimodal_dict = {}

    # Filter out rows we have already processed
    df_to_process = df[~df['sample_id'].isin(multimodal_dict.keys())]
    print(f"Remaining Utterances to process: {len(df_to_process)}")
    
    if len(df_to_process) == 0:
        print("All done!")
        return

    print("\nBeginning Multithreaded Extraction (5 Workers)...")
    # MULTITHREADING: Run 5 API calls simultaneously
    
    success_count = 0
    with ThreadPoolExecutor(max_workers=5) as executor:
        # Submit all rows to the executor
        future_to_id = {executor.submit(process_single_row, row): row['sample_id'] for _, row in df_to_process.iterrows()}
        
        # Process as they complete
        for future in tqdm(as_completed(future_to_id), total=len(future_to_id), desc="Extracting Embeddings"):
            sample_id, tensor = future.result()
            
            if tensor is not None:
                multimodal_dict[sample_id] = tensor
                success_count += 1
                
                # Save progress to disk every 100 successful extractions!
                if success_count % 100 == 0:
                    torch.save(multimodal_dict, OUTPUT_TENSORS)

    # Final Save
    torch.save(multimodal_dict, OUTPUT_TENSORS)
    print(f"\nSUCCESS! Multimodal Tensors saved to {OUTPUT_TENSORS}")

extract_vertex_multimodal_fast()

Loading out/master_dataset.csv...
Remaining Utterances to process: 7859

Beginning Multithreaded Extraction (5 Workers)...


Extracting Embeddings:   0%|          | 0/7859 [00:00<?, ?it/s]/home1/pkandpal/.conda/envs/calm_conflict/lib/python3.11/site-packages/vertexai/vision_models/_vision_models.py:499: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/home1/pkandpal/.conda/envs/calm_conflict/lib/python3.11/site-packages/vertexai/vision_models/_vision_models.py:632: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
Extracting Embeddings: 100%|██████████| 7859/7859 [3:53:34<00:00,  1.78s/it]  



SUCCESS! Multimodal Tensors saved to out/gemini_multimodal_embeddings_v2.pt
